# Classical ETL Pipeline

## Import Libraries

In [ ]:
%matplotlib inline

import sys, os, json
from datetime import datetime
from pandas.tseries.holiday import USFederalHolidayCalendar

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.loader import load_tables
from utils.schema_guard import SchemaContract
from utils.etl_pipeline import ETLPipeline

os.makedirs(DW_DATA_DIR, exist_ok=True)
os.makedirs(ETL_REPORT_DIR, exist_ok=True)

pipeline = ETLPipeline(name="Flight-Weather DW")
LOAD_TIMESTAMP = datetime.now().replace(microsecond=0).isoformat(sep=" ")
print(f"DW output:   {DW_DATA_DIR}")
print(f"ETL reports: {ETL_REPORT_DIR}")

## Loading Cleaned Data

In [ ]:
dfs = load_tables("cleaned", normalize_cols="lower")

contract = SchemaContract()
for name, df in dfs.items():
    contract.register(name, df)

validation = contract.validate_all(dfs, strict=True)

print("\nSchema validation results:")
for table_name, result in validation["results"].items():
    print(f"  [{result['status']}] {table_name}")
    for detail in result["details"]:
        print(f"    - {detail['status']:<4} {detail['check']}: {detail['message']}")

if validation["failures"] == 0:
    print("\nAll cleaned tables match expected schemas.")
else:
    print(f"\nSchema validation completed with {validation['failures']} failure(s).")

print(f"\nRegistered {len(contract.registered_tables)} table schemas.")
for name, df in dfs.items():
    print(f"  {name:<25} {len(df):>8,} rows  x  {len(df.columns):>3} cols")

## Extract Source Tables

In [ ]:
def extract_source_tables():
    for name, df in dfs.items():
        pipeline.log_df(f"src_{name.lower()}", df)
    return len(dfs)


pipeline.run_step("extract", extract_source_tables)

## Transform: DIM_FL_DATES

In [ ]:
def build_fl_dates(flight_df=None):
    if flight_df is None:
        flight_df = dfs["Flights"]
    dates = pd.to_datetime(flight_df["fl_date"], errors="coerce").dropna().unique()
    dim = pd.DataFrame({"FL_DATE": pd.to_datetime(dates)})
    dim = dim.sort_values("FL_DATE").reset_index(drop=True)
    dim["DAY_OF_WEEK"] = dim["FL_DATE"].dt.day_name()
    dim["IS_HOLIDAY"] = False
    dim["MONTH"] = dim["FL_DATE"].dt.month.astype("Int16")
    dim["QUARTER"] = dim["FL_DATE"].dt.quarter.astype("Int16")
    dim["YEAR"] = dim["FL_DATE"].dt.year.astype("Int16")

    # Is_Holiday: US federal holidays from pandas' built-in holiday calendar.
    calendar = USFederalHolidayCalendar()
    holiday_dates = calendar.holidays(start=dim["FL_DATE"].min(), end=dim["FL_DATE"].max()).normalize()
    dim["IS_HOLIDAY"] = dim["FL_DATE"].dt.normalize().isin(holiday_dates)

    dim.insert(0, "FL_DATE_ID", range(1, len(dim) + 1))
    dim["FL_DATE"] = dim["FL_DATE"].dt.strftime("%Y-%m-%d")
    dim["LOAD_TIMESTAMP"] = LOAD_TIMESTAMP
    return dim[["FL_DATE_ID", "FL_DATE", "DAY_OF_WEEK", "IS_HOLIDAY", "MONTH", "QUARTER", "YEAR", "LOAD_TIMESTAMP"]]


fl_dates = pipeline.run_step("build_fl_dates", build_fl_dates)
pipeline.log_df("DIM_FL_DATES", fl_dates)
fl_dates.head()

## Transform: DIM_DEP_HOURS

In [ ]:
def build_dep_hours(flight_df=None):
    if flight_df is None:
        flight_df = dfs["Flights"]
    hours = sorted(flight_df["dep_hour"].dropna().unique())
    dim = pd.DataFrame({"DEP_HOUR": [int(h) for h in hours]})

    def time_band(h):
        if 0 <= h < 4:
            return "Night"
        if 4 <= h < 8:
            return "Early Morning"
        if 8 <= h < 12:
            return "Morning"
        if 12 <= h < 16:
            return "Afternoon"
        if 16 <= h < 20:
            return "Evening"
        return "Late Evening"

    dim["TIME_BAND"] = dim["DEP_HOUR"].apply(time_band)
    dim.insert(0, "DEP_HOUR_ID", range(1, len(dim) + 1))
    dim["LOAD_TIMESTAMP"] = LOAD_TIMESTAMP
    return dim[["DEP_HOUR_ID", "DEP_HOUR", "TIME_BAND", "LOAD_TIMESTAMP"]]


dep_hours = pipeline.run_step("build_dep_hours", build_dep_hours)
pipeline.log_df("DIM_DEP_HOURS", dep_hours)
dep_hours.head()

## Transform: DIM_STATIONS

In [ ]:
def build_stations(station_df=None):
    if station_df is None:
        station_df = dfs["Stations"]
    cols = ["airport", "display_airport_name", "display_airport_city_name_full", "airport_state_name", "latitude", "longitude"]
    if "airport_state_code" in station_df.columns:
        cols.append("airport_state_code")
    dim = station_df[cols].copy()

    # Normalize city names by removing a trailing state suffix (e.g., "Petersburg, Ak" -> "Petersburg").
    def _clean_city(city, state_name=None, state_code=None):
        if pd.isna(city):
            return city
        raw = str(city).strip()
        if "," not in raw:
            return raw
        base, suffix = raw.rsplit(",", 1)
        suffix_norm = suffix.strip().replace(".", "").lower()
        state_norm = str(state_name).strip().replace(".", "").lower() if state_name is not None else ""
        code_norm = str(state_code).strip().replace(".", "").lower() if state_code is not None else ""
        if suffix_norm and suffix_norm in {state_norm, code_norm}:
            return base.strip()
        return raw

    dim["display_airport_city_name_full"] = dim.apply(
        lambda r: _clean_city(
            r["display_airport_city_name_full"],
            r.get("airport_state_name"),
            r.get("airport_state_code"),
        ),
        axis=1,
    )
    if "airport_state_code" in dim.columns:
        dim = dim.drop(columns=["airport_state_code"])

    dim = dim.drop_duplicates(subset=["airport"]).reset_index(drop=True)
    dim.insert(0, "STATION_ID", range(1, len(dim) + 1))
    dim = dim.rename(
        columns={
            "airport": "AIRPORT_CODE",
            "display_airport_name": "AIRPORT_NAME",
            "display_airport_city_name_full": "CITY",
            "airport_state_name": "STATE",
            "latitude": "LATITUDE",
            "longitude": "LONGITUDE",
        }
    )
    dim["LOAD_TIMESTAMP"] = LOAD_TIMESTAMP
    return dim[["STATION_ID", "AIRPORT_CODE", "AIRPORT_NAME", "CITY", "STATE", "LATITUDE", "LONGITUDE", "LOAD_TIMESTAMP"]]


stations = pipeline.run_step("build_stations", build_stations)
pipeline.log_df("DIM_STATIONS", stations)
stations.head()

## Transform: DIM_CARRIERS

In [ ]:
def build_carriers(carrier_df=None):
    if carrier_df is None:
        carrier_df = dfs["Carriers"]
    dim = carrier_df[["code", "description"]].copy()
    dim = dim.drop_duplicates(subset=["code"]).reset_index(drop=True)
    dim.insert(0, "CARRIER_ID", range(1, len(dim) + 1))
    dim = dim.rename(columns={"code": "CODE", "description": "DESCRIPTION"})
    dim["LOAD_TIMESTAMP"] = LOAD_TIMESTAMP
    return dim[["CARRIER_ID", "CODE", "DESCRIPTION", "LOAD_TIMESTAMP"]]


carriers = pipeline.run_step("build_carriers", build_carriers)
carriers_dw = carriers.drop(columns=["CODE"]) if "CODE" in carriers.columns else carriers
pipeline.log_df("DIM_CARRIERS", carriers_dw)
carriers_dw.head()

## Transform: DIM_AIRCRAFTS

In [ ]:
def build_aircraft(aircraft_df=None, flight_df=None):
    if aircraft_df is None:
        aircraft_df = dfs["Aircrafts"]
    if flight_df is None:
        flight_df = dfs["Flights"]

    # Calculate aircraft age as of the reference year (2022).
    reference_year = pd.to_datetime(flight_df["fl_date"]).dt.year.max()

    ac = aircraft_df[["tail_num", "manufacturer", "year_of_manufacture"]].copy()
    ac["aircraft_age"] = reference_year - ac["year_of_manufacture"]
    ac = ac.drop(columns=["year_of_manufacture"])

    ac = ac.drop_duplicates(subset=["tail_num"]).reset_index(drop=True)
    ac.insert(0, "AIRCRAFT_ID", range(1, len(ac) + 1))
    ac = ac.rename(
        columns={
            "tail_num": "TAIL_NUM",
            "manufacturer": "MANUFACTURER",
            "aircraft_age": "AIRCRAFT_AGE",
        }
    )
    ac["LOAD_TIMESTAMP"] = LOAD_TIMESTAMP
    return ac[["AIRCRAFT_ID", "TAIL_NUM", "MANUFACTURER", "AIRCRAFT_AGE", "LOAD_TIMESTAMP"]]


aircraft = pipeline.run_step("build_aircraft", build_aircraft)
pipeline.log_df("DIM_AIRCRAFTS", aircraft)
aircraft.head()

## Transform: DIM_JUNK

In [ ]:
def build_junk(flight_df=None):
    if flight_df is None:
        flight_df = dfs["Flights"]

    cn = dfs.get("Cancellation", pd.DataFrame())
    aw = dfs.get("Active_Weather", pd.DataFrame())
    wo = dfs.get("Weather_Observations", pd.DataFrame())

    cn_reason = cn.set_index("status")["cancellation_reason"] if "status" in cn.columns else {}
    aw_desc = aw.set_index("status")["weather_description"] if "status" in aw.columns else {}

    base = pd.DataFrame(index=flight_df.index)
    base["cancelled"] = flight_df.get("cancelled")
    base["weather_observation_id"] = flight_df.get("weather_observation_id")
    base["_cancel_reason"] = base["cancelled"].map(cn_reason)

    if "obs_id" in wo.columns and "active_weather_status" in wo.columns:
        wo_status = wo.set_index("obs_id")["active_weather_status"]
        base["_aw_status"] = base["weather_observation_id"].map(wo_status)
        base["_weather_desc"] = base["_aw_status"].map(aw_desc)
    else:
        base["_weather_desc"] = None

    dim = base[["_cancel_reason", "_weather_desc"]].drop_duplicates()
    dim = dim.rename(
        columns={
            "_cancel_reason": "CANCELLATION_REASON",
            "_weather_desc": "WEATHER_DESCRIPTION",
        }
    )
    dim = dim.sort_values(dim.columns.tolist(), na_position="last").reset_index(drop=True)
    dim.insert(0, "JUNK_ID", range(1, len(dim) + 1))
    dim["LOAD_TIMESTAMP"] = LOAD_TIMESTAMP
    return dim[["JUNK_ID", "CANCELLATION_REASON", "WEATHER_DESCRIPTION", "LOAD_TIMESTAMP"]]


junk = pipeline.run_step("build_junk", build_junk)
pipeline.log_df("DIM_JUNK", junk)
junk.head(10)

## Transform: FLIGHTS (Fact Table)

In [ ]:
def build_fact_flight(
    flight_df=None,
    _fl_dates=None,
    _dep_hours=None,
    _stations=None,
    _carriers=None,
    _aircraft=None,
    _junk=None,
):
    if flight_df is None:
        flight_df = dfs["Flights"]
    if _fl_dates is None:
        _fl_dates = fl_dates
    if _dep_hours is None:
        _dep_hours = dep_hours
    if _stations is None:
        _stations = stations
    if _carriers is None:
        _carriers = carriers
    if _aircraft is None:
        _aircraft = aircraft
    if _junk is None:
        _junk = junk

    fact = flight_df.copy()

    env_measures = ["wind_spd", "wind_gust", "visibility", "temperature", "cloud_cover"]
    perf_measures = ["dep_delay", "taxi_out", "air_time", "distance"]

    # FL_DATE_ID
    date_lookup = _fl_dates.set_index("FL_DATE")["FL_DATE_ID"]
    fact["_fl_date_str"] = pd.to_datetime(fact["fl_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    fact["FL_DATE_ID"] = fact["_fl_date_str"].map(date_lookup)

    # DEP_HOUR_ID
    hour_lookup = _dep_hours.set_index("DEP_HOUR")["DEP_HOUR_ID"]
    fact["DEP_HOUR_ID"] = fact["dep_hour"].map(hour_lookup)

    # ORIGIN_STATION_ID / DEST_STATION_ID (role-playing)
    station_lookup = _stations.set_index("AIRPORT_CODE")["STATION_ID"]
    fact["ORIGIN_STATION_ID"] = fact["origin"].map(station_lookup)
    fact["DEST_STATION_ID"] = fact["dest"].map(station_lookup)

    # MKT_CARRIER_ID / OP_CARRIER_ID (role-playing)
    carrier_lookup = _carriers.set_index("CODE")["CARRIER_ID"]
    fact["MKT_CARRIER_ID"] = fact["mkt_unique_carrier"].map(carrier_lookup)
    fact["OP_CARRIER_ID"] = fact["op_unique_carrier"].map(carrier_lookup)

    # AIRCRAFT_ID
    aircraft_lookup = _aircraft.set_index("TAIL_NUM")["AIRCRAFT_ID"]
    fact["AIRCRAFT_ID"] = fact.get("tail_num", pd.Series(dtype=str)).map(aircraft_lookup)

    # JUNK_ID - resolve from CANCELLATION_REASON + WEATHER_DESCRIPTION
    cn_reason = dfs.get("Cancellation", pd.DataFrame()).set_index("status")["cancellation_reason"] if "Cancellation" in dfs else {}
    aw_desc = dfs.get("Active_Weather", pd.DataFrame()).set_index("status")["weather_description"] if "Active_Weather" in dfs else {}

    # Resolve weather_description and environmental measures through Weather_Observations -> Active_Weather.
    if "Weather_Observations" in dfs and "obs_id" in dfs["Weather_Observations"].columns:
        wo = dfs["Weather_Observations"].set_index("obs_id")
        obs_id = fact.get("weather_observation_id", pd.Series(dtype=float))
        if "active_weather_status" in wo.columns:
            fact["_aw_status"] = obs_id.map(wo["active_weather_status"])
            fact["_weather_desc"] = fact["_aw_status"].map(aw_desc)
        else:
            fact["_weather_desc"] = None
        for col in env_measures:
            if col in wo.columns:
                fact[col.upper()] = obs_id.map(wo[col])
    else:
        fact["_weather_desc"] = None

    fact["_cancel_reason"] = fact.get("cancelled", pd.Series(dtype=float)).map(cn_reason)

    # Build JUNK_ID lookup (CANCELLATION_REASON + WEATHER_DESCRIPTION)
    junk_lookup = {}
    for _, row in _junk.iterrows():
        key = (row["CANCELLATION_REASON"], row["WEATHER_DESCRIPTION"])
        junk_lookup[key] = row["JUNK_ID"]

    def resolve_junk_id(row):
        key = (row.get("_cancel_reason"), row.get("_weather_desc"))
        if key in junk_lookup:
            return junk_lookup[key]
        return junk_lookup.get((None, None))

    fact["JUNK_ID"] = fact.apply(resolve_junk_id, axis=1)

    for col in perf_measures:
        if col in fact.columns:
            fact[col.upper()] = fact[col]

    fk_cols = [
        "FL_DATE_ID",
        "DEP_HOUR_ID",
        "ORIGIN_STATION_ID",
        "DEST_STATION_ID",
        "MKT_CARRIER_ID",
        "OP_CARRIER_ID",
        "AIRCRAFT_ID",
        "JUNK_ID",
    ]
    measure_cols = [c.upper() for c in env_measures + perf_measures]

    fact = fact[fk_cols + measure_cols].copy()
    fact.insert(0, "FLIGHT_ID", range(1, len(fact) + 1))
    fact["LOAD_TIMESTAMP"] = LOAD_TIMESTAMP

    return fact


fact_flight = pipeline.run_step("build_fact_flight", build_fact_flight)
pipeline.log_df("FLIGHTS", fact_flight)
fact_flight.head()

In [ ]:
# FK completeness check
fk_cols = ["FL_DATE_ID", "DEP_HOUR_ID", "ORIGIN_STATION_ID", "DEST_STATION_ID", "MKT_CARRIER_ID", "OP_CARRIER_ID", "AIRCRAFT_ID", "JUNK_ID"]

print("FK completeness in FLIGHTS:")
fk_issues = []
for col in fk_cols:
    if col not in fact_flight.columns:
        print(f"  [SKIP] {col} not found")
        fk_issues.append(f"Missing FK column: {col}")
        continue
    null_count = fact_flight[col].isna().sum()
    total = len(fact_flight)
    pct = (1 - null_count / total) * 100 if total > 0 else 0
    status = "[OK]" if null_count == 0 else "[FAIL]"
    print(f"  {status} {col:<25} {pct:6.2f}% non-null  ({null_count:,} nulls)")
    if null_count != 0:
        fk_issues.append(f"{col}: {null_count:,} null FK values")

if fk_issues:
    raise ValueError("FK completeness validation failed: " + "; ".join(fk_issues))

## Write Data Warehouse Tables

In [ ]:
def load_dw_tables():
    dw_tables = {
        "DIM_FL_DATES": fl_dates,
        "DIM_DEP_HOURS": dep_hours,
        "DIM_STATIONS": stations,
        "DIM_CARRIERS": carriers_dw,
        "DIM_AIRCRAFTS": aircraft,
        "DIM_JUNK": junk,
        "FLIGHTS": fact_flight,
    }
    for name, df in dw_tables.items():
        path = DW_DATA_DIR / f"{name}.csv"
        df.to_csv(path, index=False)
        print(f"  [SAVED] {name:<20} -> {path}  ({len(df):,} rows)")
    return len(dw_tables)


pipeline.run_step("load_dw", load_dw_tables)

## Pipeline Summary and Log

In [ ]:
summary_df = pipeline.summary()
summary_df.to_csv(ETL_REPORT_DIR / "etl_pipeline_summary.csv", index=False)
artifact_log = pd.DataFrame(pipeline.log_entries)
artifact_log.to_csv(ETL_REPORT_DIR / "etl_artifact_log.csv", index=False)

print("\n--- ETL Pipeline Summary ---")
print(summary_df.to_string(index=False))

## Sample Analytical Query

In [ ]:
origin_station = stations.rename(
    columns={
        "STATION_ID": "ORIGIN_STATION_ID",
        "AIRPORT_CODE": "ORIGIN_AIRPORT_CODE",
        "CITY": "ORIGIN_CITY",
        "STATE": "ORIGIN_STATE",
    }
)

analytic = fact_flight.merge(origin_station[["ORIGIN_STATION_ID", "ORIGIN_STATE"]], on="ORIGIN_STATION_ID", how="left").merge(fl_dates[["FL_DATE_ID", "MONTH"]], on="FL_DATE_ID", how="left")

result = (
    analytic.groupby(["ORIGIN_STATE", "MONTH"])
    .agg(
        avg_dep_delay=("DEP_DELAY", "mean"),
        avg_distance=("DISTANCE", "mean"),
        n_flights=("FLIGHT_ID", "count"),
    )
    .reset_index()
    .sort_values(["ORIGIN_STATE", "MONTH"])
)

print("Average departure delay by origin state and month:")
print(result.head(20).to_string(index=False))
result.to_csv(ETL_REPORT_DIR / "sample_query_delay_by_state_month.csv", index=False)

In [ ]:
# Top 10 states by average departure delay
top_states = result.groupby("ORIGIN_STATE").agg(avg_dep_delay=("avg_dep_delay", "mean"), n_flights=("n_flights", "sum")).reset_index().sort_values("avg_dep_delay", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(top_states["ORIGIN_STATE"], top_states["avg_dep_delay"], color="#C0392B", alpha=0.85, height=0.5)

ax.set_title("Top 10 States by Average Departure Delay (minutes)", fontweight="bold", color="#2C3E50", loc="left", pad=15)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.3, bar.get_y() + bar.get_height() / 2, f"{w:.1f}", va="center", color="#444444", fontsize=10, fontweight="bold")

ax.invert_yaxis()
ax.xaxis.set_visible(False)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="y", length=0, labelsize=10, colors="#333333")

plt.tight_layout()
plt.savefig(ETL_REPORT_DIR / "plot_top_states_delay.png", dpi=300, bbox_inches="tight")
plt.show()

## DW Schema Overview and Metadata Export

In [ ]:
dw_tables = {
    "DIM_FL_DATES": fl_dates,
    "DIM_DEP_HOURS": dep_hours,
    "DIM_STATIONS": stations,
    "DIM_CARRIERS": carriers_dw,
    "DIM_AIRCRAFTS": aircraft,
    "DIM_JUNK": junk,
    "FLIGHTS": fact_flight,
}

schema_path = MODELING_REPORT_DIR / "schema.json"
if schema_path.exists():
    with open(schema_path, "r", encoding="utf-8") as f:
        modeling_schema = json.load(f)
    expected_tables = set(modeling_schema["star_schema"]["dimension_tables"].keys()) | {modeling_schema["star_schema"]["fact_table"]["name"]}
    actual_tables = set(dw_tables.keys())
    if expected_tables != actual_tables:
        raise ValueError(f"DW table set does not match modeling schema. Expected {sorted(expected_tables)}, got {sorted(actual_tables)}")
    print(f"DW table set matches modeling schema: {sorted(actual_tables)}")

schema_info = []
for name, df in dw_tables.items():
    pk_col = df.columns[0]
    schema_info.append(
        {
            "Table": name,
            "PK": pk_col,
            "Rows": len(df),
            "Columns": ", ".join(df.columns.tolist()),
        }
    )

schema_df = pd.DataFrame(schema_info)
schema_df.to_csv(ETL_REPORT_DIR / "dw_schema_overview.csv", index=False)

print("\n=== Data Warehouse Schema ===")
for _, row in schema_df.iterrows():
    print(f"  {row['Table']:<15} PK={row['PK']:<15} {row['Rows']:>8,} rows")

dw_metadata = {
    "generated_at": datetime.now().isoformat(),
    "pipeline_name": pipeline.name,
    "steps_completed": len([s for s in pipeline.steps if s.status == "done"]),
    "tables": {name: {"rows": len(df), "columns": df.columns.tolist(), "primary_key": df.columns[0]} for name, df in dw_tables.items()},
}
with open(ETL_REPORT_DIR / "dw_metadata.json", "w") as f:
    json.dump(dw_metadata, f, indent=2, default=str)

print(f"\nDW metadata saved. Pipeline: {len(pipeline.steps)} steps completed.")